# 04 - QoL_Stress preprocessing

ECG decoding + NeuroKit2, EDA cleaning, Fitbit phase windows, QC. Code: `src/preprocessing/qol_preprocess.py`.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)
M = ROOT / 'results' / 'metrics'; F = ROOT / 'results' / 'figures'; P = ROOT / 'data' / 'processed'
from src.preprocessing.qol_preprocess import build_qol, MODALITIES
out = build_qol(save=False)
f = out['features']

## Every exclusion, with its reason

In [2]:
out['qc'].query('included == False')

,pid,phase,modality,included,reason,detected_hr,device_hr
264,Q01,relax,HR,False,only 28 Fitbit samples in the 8-min window,NaN,NaN
256,Q01,stress,EDA,False,no EDA session in release,NaN,NaN
115,Q02,stress,ECG,False,detected HR 76.3 vs device 65 bpm,76.282940,65.0
83,Q07,stress,ECG,False,31 % of RR intervals rejected as artefacts,68.564305,69.0
257,Q15,relax,EDA,False,no EDA session in release,NaN,NaN
258,Q15,stress,EDA,False,no EDA session in release,NaN,NaN
17,Q16,stress,ECG,False,22 % of RR intervals rejected as artefacts,70.955535,68.0
15,Q20,stress,ECG,False,24 % of RR intervals rejected as artefacts,66.032752,64.0
109,Q22,stress,ECG,False,45 % of RR intervals rejected as artefacts,75.693860,73.0
1,Q28,stress,ECG,False,59 % of RR intervals rejected as artefacts,53.254438,46.0


## Modality availability (rows with the modality, and participants with it in both phases)

In [3]:
pd.DataFrame({m: {'rows': int(f[f'has_{m}'].sum()), 'participants_both_phases': int(f.loc[f[f'pair_{m}'], 'pid'].nunique())} for m in MODALITIES}).T

,rows,participants_both_phases
ECG,115,51
EDA,124,60
HR,129,64


In [4]:
print('complete-case participants:', f.loc[f.complete_case, 'pid'].nunique())

complete-case participants: 45


## Phase medians

In [5]:
f.groupby('phase')[[c for cols in MODALITIES.values() for c in cols]].median().T.round(2)

phase,relax,stress
ecg_hr,67.38,68.37
ecg_rmssd,50.09,41.50
ecg_sdnn,56.50,63.06
ecg_pnn50,27.13,22.86
ecg_hr_range,16.63,20.05
eda_scl_mean,40.73,39.23
eda_scl_std,0.81,1.07
eda_scl_slope,-0.16,-0.38
eda_scl_range,1.81,2.62
eda_resp_rate,8.57,8.30
